# GPA778 - Laboratoire 2
# Partie A - Perception I : Traitement d'image et Lidar

**Objectif :** Développer le package `jerro_perception` pour intégrer les systèmes de perception du robot (caméra 2D et Lidar) et créer un launch file pour démarrer l'ensemble des nodes.

## Objectifs du laboratoire

Dans ce laboratoire, vous allez développer deux nodes ROS 2 pour le package **jerro_perception**:

| Node | Description | Jalon |
|------|-------------|-------|
| `cam_traitement` | Applique un filtre de détection de contours (Canny) sur les images de la caméra | Jalon 1 |
| `lidar_traitement` | Convertit les données LaserScan en coordonnées cartésiennes et les publie en PointCloud2 | Jalon 2 |

Vous allez aussi créer un **launch file** pour démarrer tous les nodes en une seule commande (Jalon 3).

### Ce que vous allez apprendre:
- Créer un package ROS 2 Python
- Utiliser **cv_bridge** pour convertir les messages ROS en images OpenCV
- Appliquer le filtre de Canny pour la détection de contours
- Traiter les données du Lidar (conversion polaire → cartésien)
- Écrire un launch file ROS 2

## Filtrer une image à partir de la caméra

### Paramétrer la caméra

Jerro a une caméra fixée à son avant, elle est branchée via CSI au Raspberry Pi. Pour obtenir les images sur un topic ROS 2, on utilise le package [`v4l2_camera`](https://docs.ros.org/en/rolling/p/v4l2_camera/):
```bash
ros2 run v4l2_camera v4l2_camera_node --ros-args -p image_raw.enable_pub_plugins:="['image_transport/raw','image_transport/compressed']"
# Les erreurs affichées sont "normales"
```

Ce package se connecte au flux de données de la caméra et le publie sur les topics suivants:
```bash
ubuntu@ubuntu:~/ros2_ws$ ros2 topic list
/camera_info  # Les paramètres de calibration de la caméra
/image_raw    # L'image qui provient directement de la caméra
```

Pour plus d'information sur ces messages: [Image](https://docs.ros.org/en/noetic/api/sensor_msgs/html/msg/Image.html), [CameraInfo](https://docs.ros.org/en/melodic/api/sensor_msgs/html/msg/CameraInfo.html).

### Fichier de calibration

En écoutant le topic `/camera_info`, vous remarquerez que tous les paramètres sont à 0. Cela provient de l'erreur suivante au démarrage du node:
```bash
[ERROR] [camera_calibration_parsers]: Unable to open camera calibration file [/home/ubuntu/.ros/camera_info/mmal_service_16.1.yaml]
```

Pour corriger cette erreur, copiez le fichier `mmal_service_16.1.yaml` fourni sur Moodle dans le dossier `/home/ubuntu/.ros/camera_info/`. Après redémarrage du node, le topic `/camera_info` devrait contenir les valeurs de calibration.

### Création du package

Dans votre workspace, créez le package `jerro_perception` avec les dépendances nécessaires:
```bash
ros2 pkg create --build-type ament_python jerro_perception \
    --dependencies rclpy sensor_msgs std_msgs cv_bridge sensor_msgs_py
```

### Conversion vers OpenCV

La librairie `cv_bridge` permet de convertir les images ROS 2 en images [OpenCV](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html) (et vice-versa). OpenCV est une librairie qui offre de nombreuses opérations de traitement d'image.

Pour importer la librairie:
```python
from cv_bridge import CvBridge
```

Dans l'initialisation du node:
```python
self.bridge = CvBridge()
```

Pour convertir les images:
```python
# Message ROS 2 vers OpenCV
cv_image = self.bridge.imgmsg_to_cv2(msg, "bgr8")

# OpenCV vers message ROS 2
ros_image = self.bridge.cv2_to_imgmsg(cv_image, encoding="mono8")
```

### Le filtre Canny

Le filtre Canny est un algorithme de détection de contours. Pour l'appliquer, importez OpenCV:
```python
import cv2
```



```python
# 2. Appliquer le filtre Canny
img_canny = cv2.Canny(cv_image, t_lower, t_upper)
```

Les paramètres `t_lower` et `t_upper` contrôlent les seuils de détection du filtre (ex: 50 et 150).

## Utilisation du lidar

### Installation du package sllidar_ros2

Le lidar de Jerro est un SLAMTEC S2. Pour obtenir les données sur un topic ROS 2, il faut installer le package `sllidar_ros2`:
```bash
cd ~/ros2_ws/src
git clone https://github.com/Slamtec/sllidar_ros2.git
cd ~/ros2_ws
colcon build --packages-select sllidar_ros2
source install/setup.bash
```

Pour lancer le driver du lidar:
```bash
ros2 launch sllidar_ros2 sllidar_s2_launch.py
```

Le driver publie les données sur le topic `/scan` avec le type `sensor_msgs/msg/LaserScan`. Pour plus d'information sur ce message: [LaserScan](https://docs.ros2.org/latest/api/sensor_msgs/msg/LaserScan.html).

### Conversion des données LaserScan

Le message LaserScan contient les distances mesurées en coordonnées polaires (angle + distance). Pour les convertir en coordonnées cartésiennes (x, y, z), une fonction vous est fournie sur Moodle: `laser_scan_to_array.py`.

Copiez ce fichier dans votre package:
```bash
cp laser_scan_to_array.py ~/ros2_ws/src/jerro_perception/jerro_perception/
```

Pour utiliser cette fonction dans votre node:
```python
from jerro_perception.laser_scan_to_array import laser_scan_to_array

# Dans le callback du subscriber
points = laser_scan_to_array(msg, remove_invalid_ranges=True)
# points est un tableau numpy de shape (N, 3) avec colonnes [x, y, z]
```

Le paramètre `remove_invalid_ranges=True` permet de filtrer les valeurs invalides (`inf`, `nan`) qui peuvent apparaitre lorsque le lidar ne détecte pas d'obstacle.

### Publication sur PointCloud2

Pour publier les points convertis, utilisez le message `sensor_msgs/msg/PointCloud2`. La librairie `sensor_msgs_py` fournit des fonctions utilitaires pour créer ce message:

```python
from sensor_msgs.msg import PointCloud2
from sensor_msgs_py import point_cloud2

# Dans le callback, après la conversion
points = laser_scan_to_array(msg, remove_invalid_ranges=True)

# Créer le message PointCloud2
cloud_msg = point_cloud2.create_cloud_xyz32(msg.header, points)

# Publier sur /points
self.publisher.publish(cloud_msg)
```

Le `msg.header` est réutilisé pour conserver le timestamp et le frame_id du message original.

## Création d'un Launch File

Un launch file permet de démarrer plusieurs nodes en une seule commande. Dans ROS 2, les launch files sont écrits en Python.

### Structure du dossier launch

Créez un dossier `launch` dans votre package:
```bash
mkdir ~/ros2_ws/src/jerro_perception/launch
```

### Écriture du launch file

Créez le fichier `perception_launch.py` dans le dossier `launch`. Voici la structure de base d'un launch file ROS 2:

```python
from launch import LaunchDescription
from launch_ros.actions import Node

def generate_launch_description():
    return LaunchDescription([
        # Ajoutez vos nodes ici
    ])
```

### Ajouter des nodes avec paramètres

Pour ajouter un node dans le launch file, utilisez la classe `Node`:
```python
Node(
    package='nom_du_package',
    executable='nom_executable',
    name='nom_du_node',
    parameters=[{
        'param1': valeur1,
        'param2': valeur2
    }]
)
```

#### Exemple: Node v4l2_camera avec paramètres

Le node de la caméra nécessite un paramètre spécial. Rappelez-vous que nous le lancions avec:
```bash
ros2 run v4l2_camera v4l2_camera_node --ros-args -p image_raw.enable_pub_plugins:="['image_transport/raw','image_transport/compressed']"
```

Pour traduire cette commande en launch file, le nom du paramètre reste le même (avec le point) et la valeur est une chaîne de caractères:

```python
Node(
    package='v4l2_camera',
    executable='v4l2_camera_node',
    name='v4l2_camera_node',
    parameters=[{
        'image_raw.enable_pub_plugins': ['image_transport/raw','image_transport/compressed']
    }]
)
```

**Note:** Les paramètres avec des points dans leur nom (comme `image_raw.enable_pub_plugins`) sont valides en ROS 2 et doivent être écrits exactement comme ils apparaissent dans la commande d'origine.

**À faire:** Ajoutez les 3 nodes nécessaires dans votre launch file:
1. Le node `v4l2_camera_node` du package `v4l2_camera` (avec le paramètre ci-dessus)
2. Le node `cam_traitement` de votre package
3. Le node `lidar_traitement` de votre package

### Inclure un launch file existant

Pour inclure un launch file d'un autre package (comme celui du lidar), utilisez `IncludeLaunchDescription`. Cela permet de réutiliser des launch files existants sans dupliquer leur contenu.

**Imports supplémentaires nécessaires:**
```python
from launch.actions import IncludeLaunchDescription
from launch.launch_description_sources import PythonLaunchDescriptionSource
from ament_index_python.packages import get_package_share_directory
```

**Exemple pour inclure le launch file du lidar:**
```python
IncludeLaunchDescription(
    PythonLaunchDescriptionSource([
        get_package_share_directory('sllidar_ros2'),
        '/launch/sllidar_s2_launch.py'
    ])
)
```

La fonction `get_package_share_directory()` retourne le chemin vers le dossier `share` du package installé. Cela permet de trouver le launch file peu importe où le workspace est installé.

**À faire:** Incluez le launch file du lidar dans votre `perception_launch.py` pour que tout démarre avec une seule commande.

### Configuration de setup.py

Pour que le launch file soit installé avec le package, vous devez modifier le fichier `setup.py`.

**1. Ajoutez les imports en haut du fichier:**
```python
import os
from glob import glob
```

**2. Dans la liste `data_files`, ajoutez cette ligne:**
```python
(os.path.join('share', package_name, 'launch'), glob('launch/*.py')),
```

**3. Dans `entry_points`, ajoutez vos deux nodes:**
```python
entry_points={
    'console_scripts': [
        'cam_traitement = jerro_perception.cam_traitement:main',
        'lidar_traitement = jerro_perception.lidar_traitement:main',
    ],
},
```

### Lancement

Après avoir compilé le package, lancez le launch file avec:
```bash
cd ~/ros2_ws
colcon build --packages-select jerro_perception
source install/setup.bash
ros2 launch jerro_perception perception_launch.py
```

Tous les nodes devraient démarrer en même temps. Vous pouvez vérifier avec:
```bash
ros2 node list
```

## Vérification

Utilisez les commandes suivantes pour vérifier que vos nodes fonctionnent correctement:

```bash
# Lister les topics actifs
ros2 topic list

# Vérifier que /image_canny est publié
ros2 topic hz /image_canny

# Vérifier que /points est publié
ros2 topic hz /points

# Lister les nodes actifs
ros2 node list
```

**Visualisation avec Rviz:**
1. Connectez-vous au robot via Rviz Studio
2. Ajoutez un panneau **Image** pour visualiser `/image_raw` et `/image_canny` côte à côte
3. Ajoutez un panneau **point_cloud2** pour visualiser le nuage de points `/points`

## Démonstration au chargé de laboratoire

À la fin de ce laboratoire, vous devez démontrer:

| Jalon | Démonstration attendue |
|-------|------------------------|
| 1 | Le node `cam_traitement` publie les contours détectés sur `/image_canny` |
| 2 | Le node `lidar_traitement` publie les points convertis sur `/points` |
| 3 | Le launch file démarre tous les nodes avec une seule commande |